# 🧪 Probar el asistente de reservas (Gemma 4 E2B fine-tuned) + `backend_sim`

Este cuaderno **carga tu adapter ya entrenado** y te deja **charlar con el modelo**.
Cuando el modelo necesita datos (disponibilidad, estado de una reserva…) emite una
llamada a herramienta y **`backend_sim` la ejecuta de verdad** y le devuelve el
resultado. Es el bucle completo *modelo ↔ datos*, con un backend simulado.

**Necesitas:**
- `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
- Nada más: el **adapter** entrenado ya viene en el repo y el Paso 2 lo trae con el `git clone`.

> 💡 Ejecútalo en el **navegador** (no headless): la sesión aguanta sin desconexiones.

## Paso 1 · Instalar Unsloth

In [ ]:
%%capture
!pip install unsloth

## Paso 2 · Traer `backend_sim` (el backend simulado)
Lo clonamos del repo; trae también el catálogo y el system prompt.

In [ ]:
!git clone --depth 1 https://github.com/noelserdna/capa1-ingesta-colab.git
%cd capa1-ingesta-colab/fine
import sys
sys.path.append("data")     # para "import backend_sim"
import backend_sim as B
print("backend_sim OK · instalaciones:", list(B.CATALOGO)[:5], "...")

## Paso 3 · Apuntar al adapter entrenado

El adapter entrenado **viene incluido en el repo** que clonaste en el Paso 2, así que
solo hay que apuntar a su carpeta. (Si has entrenado el tuyo propio, usa las
alternativas comentadas: subirlo como `adapter.zip` o cargarlo desde Drive.)

In [ ]:
import os
RUTA_ADAPTER = "/content/capa1-ingesta-colab/fine/adapter"
print("Adapter:", RUTA_ADAPTER, "->", os.listdir(RUTA_ADAPTER))

# --- Alternativa B: subir TU adapter.zip (cd fine && zip -r adapter.zip adapter) ---
# from google.colab import files
# import zipfile, io
# subido = files.upload()
# with zipfile.ZipFile(io.BytesIO(next(iter(subido.values())))) as z: z.extractall("/content")
# RUTA_ADAPTER = "/content/adapter"

# --- Alternativa C: desde Google Drive ---
# from google.colab import drive; drive.mount("/content/drive")
# RUTA_ADAPTER = "/content/drive/MyDrive/gemma4_reservas/adapter"

## Paso 4 · Cargar el modelo + adapter

`FastModel.from_pretrained(RUTA_ADAPTER)` descarga la base Gemma 4 E2B (4-bit) y le
aplica tu adapter LoRA. Ojo: en Gemma 4 el `tokenizer` es un `Gemma4Processor`, así
que para generar usamos su **tokenizer interno** y le ponemos nuestra plantilla.

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=RUTA_ADAPTER, max_seq_length=2048, load_in_4bit=True)
FastModel.for_inference(model)

PLANTILLA = (
    "{{ bos_token }}"
    "{% set ns = namespace(system='') %}"
    "{% for message in messages %}"
        "{% if message['role'] == 'system' %}"
            "{% set ns.system = message['content'] %}"
        "{% elif message['role'] in ['user', 'tool'] %}"
            "{{ '<start_of_turn>user\n' }}"
            "{% if ns.system %}{{ ns.system + '\n\n' }}{% set ns.system = '' %}{% endif %}"
            "{% if message['role'] == 'tool' %}"
                "{{ '```tool_result\n' + message['content'] + '\n```' }}"
            "{% else %}"
                "{{ message['content'] }}"
            "{% endif %}"
            "{{ '<end_of_turn>\n' }}"
        "{% elif message['role'] == 'assistant' %}"
            "{{ '<start_of_turn>model\n' + message['content'] + '<end_of_turn>\n' }}"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<start_of_turn>model\n' }}{% endif %}"
)
tok = getattr(tokenizer, "tokenizer", tokenizer)   # tokenizer interno del processor
tok.chat_template = PLANTILLA
FIN = tok.convert_tokens_to_ids("<end_of_turn>")
print("Modelo listo ✅")

## Paso 5 · El bucle de herramientas

`generar_turno` produce el siguiente turno del modelo. Si es una `tool_call`, la
ejecutamos contra `backend_sim` y le devolvemos el resultado; repetimos hasta que el
modelo responde al usuario.

In [ ]:
import re, json

def extraer_tool_call(texto):
    m = re.search(r"```tool_call\s*(\{.*?\})\s*```", texto, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(1))
    except json.JSONDecodeError:
        return None

def generar_turno(messages, max_new_tokens=256):
    entrada = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(entrada, return_tensors="pt", add_special_tokens=False).to("cuda")
    salida = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.3,
                            top_p=0.95, do_sample=True,
                            eos_token_id=[tok.eos_token_id, FIN])
    texto = tok.decode(salida[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    return texto.split("<end_of_turn>")[0].strip()

def responder(backend, messages, texto_usuario, verbose=True):
    messages.append({"role": "user", "content": texto_usuario})
    for _ in range(6):
        salida = generar_turno(messages)
        messages.append({"role": "assistant", "content": salida})
        tc = extraer_tool_call(salida)
        if tc is None:
            return salida
        resultado = backend.ejecutar(tc["tool"], tc.get("args", {}))
        if verbose:
            print("   🔧 %s(%s) → %s" % (tc["tool"],
                  json.dumps(tc.get("args", {}), ensure_ascii=False),
                  json.dumps(resultado, ensure_ascii=False)))
        messages.append({"role": "tool", "content": json.dumps(resultado, ensure_ascii=False)})
    return "(se alcanzó el máximo de llamadas a herramienta)"

print("Funciones del bucle listas ✅")

## Paso 6 · Demos guiadas

Cuatro escenarios automáticos. Fíjate en las líneas `🔧`: son las llamadas reales a
`backend_sim`.

In [ ]:
def demo(backend, entradas):
    msgs = [{"role": "system", "content": B.system_prompt(backend.hoy)}]
    for txt in entradas:
        print("\n🧑 " + txt)
        print("🤖 " + responder(backend, msgs, txt))

print("################ Reservar pádel ################")
demo(B.Backend(hoy="2026-09-01", seed=123),
     ["Hola, quiero reservar una pista de pádel para mañana por la tarde",
      "a las 7, una hora", "seremos 4", "Andrés López", "sí"])

print("\n################ Consultar disponibilidad ################")
demo(B.Backend(hoy="2026-09-01", seed=5),
     ["¿hay tenis libre el sábado por la mañana?"])

print("\n################ Off-topic + estado de reserva ################")
bk = B.Backend(hoy="2026-09-01", seed=7)
_hora = bk.buscar_huecos("padel", "2026-09-05", 60, max_op=1)[0]["hora"]  # hueco libre real
r = bk.crear_reserva("padel", "2026-09-05", _hora, 60, "Marta Ruiz", num_personas=2)
print("(reserva sembrada: %s)" % r["localizador"])
demo(bk, ["oye, ¿sabes si mañana lloverá?",
          "vale, mira a ver mi reserva", "por nombre, Marta Ruiz"])

## Paso 7 · 💬 Chat interactivo

Ejecuta y **escribe tú**. El modelo conversa y, cuando hace falta, llama a
`backend_sim`. Escribe `salir` para terminar.

> La "fecha de hoy" del asistente es la de `HOY` (cámbiala si quieres). Las reservas
> que crees se guardan en `bk` y puedes inspeccionarlas en el Paso 8.

In [ ]:
HOY = "2026-09-01"
bk = B.Backend(hoy=HOY, seed=1)
mensajes = [{"role": "system", "content": B.system_prompt(bk.hoy)}]

print("Chat con el asistente de reservas (fecha de hoy: %s). Escribe 'salir'.\n" % HOY)
while True:
    try:
        texto = input("🧑 ").strip()
    except (EOFError, KeyboardInterrupt):
        break
    if texto.lower() in ("salir", "exit", "quit", ""):
        print("¡Hasta luego!")
        break
    print("🤖 " + responder(bk, mensajes, texto))

## Paso 8 · Ver el estado del backend

Las reservas creadas durante la sesión (lo que tu app guardaría en la base de datos real).

In [ ]:
import json
if not bk.reservas:
    print("(todavía no hay reservas en este backend)")
for loc, r in bk.reservas.items():
    print(loc, "→", json.dumps(r, ensure_ascii=False))